In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
import numpy as np
import rasterio
from tqdm.notebook import tqdm
import pandas as pd
import rasterio
from datetime import datetime
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as v2
import torchvision.models as tmodels
from torch.optim.lr_scheduler import CosineAnnealingLR, MultiStepLR
from torchinfo import summary

from huggingface_hub import hf_hub_download
from terratorch.models.backbones.prithvi_mae import PrithviViT

from models import ModifiedResNet18, ModifiedPrithviResNet18, prithvi_terratorch
from utils import set_seed, f1_score
from glc_datasets import load_raster, TestDataset, TrainDataset, HorizontalCycleTransform, HorizontalPermuteTransform
from glc_datasets import read_train_data, read_test_data

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
try:
    from config import DATA_PATH, SCRATCH_PATH, PRITHVI_WEIGHTS_PATH
except ImportError:
    DATA_PATH = os.getenv('GLC_DATA_PATH')
    SCRATCH_PATH = os.getenv('GLC_SCRATCH_PATH')
    PRITHVI_WEIGHTS_PATH = os.getenv('PRITHVI_WEIGHTS_PATH')

In [ ]:
batch_size = 32
num_workers = 6
pa_presence_threshold = 1
num_classes_total = 11255
landsat_year_len = 18
bioclim_month_len = landsat_year_len*12-1
validation_prop = 0.1
sel_countries = ["France", "Denmark", "Netherlands", "Italy"] #, "Austria"
cov_countries = 1
cov_area, cov_elevation, cov_snow = 1, 1, 0
cov_soil, cov_worldcover, cov_landcover = 1, 1, 0
if os.uname()[1] == "localhostname":
    path_save = path_data = DATA_PATH
    print("local, using", f"path_data=path_save={path_data}")
else:
    path_data = os.environ.get('LOCAL_SCRATCH', DATA_PATH)
    path_save = SCRATCH_PATH
    print("hpc, using", f"path_data={path_data};", f"path_save={path_save}")
    
mean_landsat = 1*np.array([ 15.0698,   16.0923,    7.9312,   68.9794,   47.9505,   24.8804, 7089.4349, 2830.6658])
std_landsat =  1*np.array([ 11.7218,   10.2417,    9.6499,   18.7112,   13.1681,    9.2436, 3332.3618,   56.7270])
mean_sentinel = 1*np.array([ 624.8547,  684.7646,  456.7674, 2924.1753])
std_sentinel =  1*np.array([ 416.0408,  351.1005,  315.8956,  943.6141])

transform_landsat = v2.Compose([
    v2.Normalize(mean_landsat, std_landsat),
    HorizontalCycleTransform()
])
transform_landsat_test = v2.Compose([
    v2.Normalize(mean_landsat, std_landsat),
])
transform_sentinel = v2.Compose([
    v2.Normalize(mean_sentinel, std_sentinel),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.5),
    v2.RandomRotation(180),
    v2.RandomResizedCrop(size=64, scale=(0.25, 1.0))
])
transform_sentinel_test = v2.Compose([
    v2.Normalize(mean_sentinel, std_sentinel)
])

# Load training data

In [ ]:
set_seed(42)
train_path_sentinel = os.path.join(path_data, "SatelitePatches/PA-train")
train_path_landsat = os.path.join(path_data, "SateliteTimeSeries-Landsat/cubes/PA-train")
train_path_bioclim = os.path.join(path_data, "BioclimTimeSeries/cubes/PA-train")
cov_flag_list = [cov_area, cov_elevation, cov_countries, cov_soil, cov_worldcover, cov_landcover, cov_snow]
train_combined, train_label_series, sp_categories, cov_columns, cov_norm_coef, num_classes = read_train_data(path_data, cov_flag_list, sel_countries, pa_presence_threshold)

val_ind = np.sort(train_combined.surveyId.sample(frac=validation_prop).values)
train_data, val_data = [x.reset_index(drop=True) for _, x in train_combined.groupby(train_combined.surveyId.isin(val_ind))]
train_label_dict = train_label_series[train_data.surveyId].to_dict()
val_label_dict = train_label_series[val_data.surveyId].to_dict()
train_dataset = TrainDataset(train_path_sentinel, train_path_landsat, train_path_bioclim, train_data, cov_columns, train_label_dict, 
                             subset="train", num_classes=num_classes, transform_sentinel=transform_sentinel, transform_landsat=transform_landsat,
                             landsat_year_len=landsat_year_len, image_mean=False, sentinel_mask_channel=True)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_dataset = TrainDataset(train_path_sentinel, train_path_landsat, train_path_bioclim, val_data, cov_columns, val_label_dict,
                           subset="train", num_classes=num_classes, transform_sentinel=transform_sentinel_test, transform_landsat=transform_landsat_test,
                           landsat_year_len=landsat_year_len, image_mean=False, sentinel_mask_channel=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
print(train_dataset[0][0].shape, val_dataset[0][2].shape)

In [ ]:
train_combined.loc[:,cov_columns].agg(['mean', 'std'])

In [ ]:
set_seed(1)
N = np.minimum(len(train_dataset), int(1e2)) 
ind = np.sort(np.random.choice(len(train_dataset), size=N, replace=True))
sentinel_channel_sum, sentinel_channel_sum2 = torch.zeros([5], dtype=torch.float64), torch.zeros([5], dtype=torch.float64)
landsat_bioclim_channel_sum, landsat_bioclim_channel_sum2 = torch.zeros([8], dtype=torch.float64), torch.zeros([8], dtype=torch.float64)
for i in tqdm(ind):
    train_item = train_dataset[i]
    sentinel_channel_sum += torch.mean(train_item[0], [-1,-2])
    sentinel_channel_sum2 += torch.mean(train_item[0]**2, [-1,-2])
    landsat_bioclim_channel_sum += torch.mean(train_item[1], [-1,-2])
    landsat_bioclim_channel_sum2 += torch.mean(train_item[1]**2, [-1,-2])
sentinel_channel_mean = sentinel_channel_sum / N
sentinel_channel_std = np.sqrt((sentinel_channel_sum2 - sentinel_channel_sum**2) / N)
landsat_bioclim_channel_mean = landsat_bioclim_channel_sum / N
landsat_bioclim_channel_std = np.sqrt((landsat_bioclim_channel_sum2 - landsat_bioclim_channel_sum**2) / N)
print(sentinel_channel_mean, sentinel_channel_std)
print(landsat_bioclim_channel_mean, landsat_bioclim_channel_std)

In [ ]:
print("train size: ", len(train_dataset), " | validation size: ", len(val_dataset))
survey_ind = 0
plt.figure(figsize=[15,4])
plt.subplot(1, 3, 1)
plt.hist(train_dataset[survey_ind][0].flatten())
plt.title("[%.3f | %.3f]" % (np.min(train_dataset[survey_ind][0].numpy()), np.max(train_dataset[survey_ind][0].numpy())))
plt.subplot(1, 3, 2)
img_sentinel = train_dataset[survey_ind][0]
plt.imshow(torch.permute(img_sentinel[:3], [1,2,0]))
plt.subplot(1, 3, 3)
plt.imshow(img_sentinel[-1])
plt.show()
print(img_sentinel[-1])

In [ ]:
test_path_sentinel = os.path.join(path_data, "SatelitePatches/PA-test")
test_path_landsat = os.path.join(path_data, "SateliteTimeSeries-Landsat/cubes/PA-test")
test_path_bioclim = os.path.join(path_data, "BioclimTimeSeries/cubes/PA-test")
test_combined = read_test_data(path_data, cov_columns, cov_norm_coef, sel_countries)
test_combined.reset_index(drop=True, inplace=True)
test_dataset = TestDataset(test_path_sentinel, test_path_landsat, test_path_bioclim, test_combined, cov_columns, subset="test", 
                           transform_sentinel=transform_sentinel_test, transform_landsat=transform_landsat_test, image_mean=False, sentinel_mask_channel=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
print(test_dataset[0][0].shape)

In [ ]:
fig, axes = plt.subplots(2, 3, squeeze=False, sharey="all", figsize=[18,8])
id_im = 0*6
for idr, axr in enumerate(axes):
    for idc, ax in enumerate(axr):
        for ch in range(train_dataset[id_im][1].shape[0]):
            ax.plot(torch.transpose(train_dataset[id_im][1][ch], 1,0).numpy().reshape(-1))
        id_im += 1
        ax.set_title("surveyId %d"%train_dataset[id_im][-1])
fig.show()

# Setup model

In [ ]:
patch_size = [1,16,16]
n_frame = 1
n_channel = 5
embed_dim = 1024
decoder_depth = 8
num_heads = 16
mlp_ratio = 4
resnet_dim = 1000
hidden_last_dim = 1000 + 128
head_dropout = 0.0
      
path_prithvi = PRITHVI_WEIGHTS_PATH
wt_file = os.path.join(path_prithvi, "Prithvi_EO_V2_300M_TL.pt")
if not os.path.isfile(wt_file):
  hf_hub_download(repo_id="ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL", filename="Prithvi_EO_V2_300M_TL.pt", local_dir=path_prithvi)

prithvi_instance = PrithviViT(
        patch_size=patch_size,
        num_frames=n_frame,
        in_chans=n_channel,
        embed_dim=embed_dim,
        decoder_depth=decoder_depth,
        num_heads=num_heads,
        mlp_ratio=mlp_ratio,
        head_dropout=head_dropout,
        backbone_input_size=[1,64,64],
        encoder_only=False,
        padding=True,
)
prithvi_model = prithvi_terratorch(wt_file, prithvi_instance, [1,64,64])
prithvi_model.freeze_encoder()

device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("DEVICE = CUDA")
prithvi_model.to(device);

In [ ]:
set_seed(69)
model = ModifiedPrithviResNet18(num_classes, len(cov_columns), resnet_dim, hidden_last_dim, prithvi_model).to(device)

In [ ]:
# resNet18 = ModifiedResNet18()
# summary(resNet18, (32, 8, 4, 18))
# tmp = tmodels.resnet18(weights=None)
# tmp.layer4 = nn.Identity()
# tmp.fc = nn.Linear(256, 1000)
# summary(tmp, (32, 3, 4, 18))
#summary(model.landsat_part, [(4, 1, 64, 64), (6, 4, 21)])
#list(model.fc_final.named_parameters())
#[name for name, p in model.named_parameters()]

In [ ]:
n = 10
sentinel_batch = torch.stack([train_dataset[i][0] for i in range(n)])[:,:,None,:,:]
landsat_batch = torch.stack([train_dataset[i][1] for i in range(n)])
cov_batch = torch.stack([train_dataset[i][2] for i in range(n)])
lonlat_batch = torch.stack([train_dataset[i][3] for i in range(n)])
prithvi_res = prithvi_model.forward(sentinel_batch.to(device), None, lonlat_batch.to(device), 0)
print(prithvi_res.to(device).shape)
model_res = model.forward(sentinel_batch.to(device), landsat_batch.to(device), cov_batch.to(device), lonlat_batch.to(device))
print(model_res.shape)

In [ ]:
# Hyperparameters
learning_rate = 0.0002
weight_decay_tail = 1e-4
epoch_min = 10
top_res_n = 2
num_epochs = 75
positive_weigh_factor = 1.0

#optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

tail_weight_names = ['fc_final.weight', 'fc_tail.weight']
print([name for name, p in model.named_parameters() if any([par_tail_name in name for par_tail_name in tail_weight_names])])
last_layer_weights = [p for name, p in model.named_parameters() if any([par_tail_name in name for par_tail_name in tail_weight_names])]
others = [p for name, p in model.named_parameters() if not any([par_tail_name in name for par_tail_name in tail_weight_names])]
optim_dict = [{"params": others}, {"params": last_layer_weights, "weight_decay": weight_decay_tail}]
optimizer = torch.optim.AdamW(optim_dict, lr=learning_rate)
schedulerCos = CosineAnnealingLR(optimizer, T_max=24.5, verbose=True)
# schedulerMult = MultiStepLR(optimizer, milestones=[30,80], gamma=0.1)

In [ ]:
def validate():
    criterion = nn.BCEWithLogitsLoss()
    loss_array = np.zeros(len(val_loader))
    f1_25_list, f1_opt_list = [None]*len(val_loader), [None]*len(val_loader)
    for batch_idx, (data_sentinel, data_landsat, data_cov, data_lonlat, targets, _) in tqdm(enumerate(val_loader), total=len(val_loader), desc="validation", leave=False):
        data_sentinel = data_sentinel.to(device)[:,:,None,:,:]
        data_landsat, data_cov, data_lonlat, targets = data_landsat.to(device), data_cov.to(device), data_lonlat.to(device), targets.to(device)
        outputs = model(data_sentinel, data_landsat, data_cov, data_lonlat)
        loss = criterion(outputs, targets)
        loss_array[batch_idx] = loss.item()
        f1_25_list[batch_idx] = f1_score(outputs, targets, M=1, mult=0, offset=25, device=device) # torch.zeros([targets.shape[0]], device=device) #f1_score(outputs, targets, M=1, mult=0, offset=25)
        f1_opt_list[batch_idx] = f1_score(outputs, targets, M=401, mult=1, offset=0, device=device)
    mean_val_loss = np.mean(loss_array)
    mean_f1_25, mean_f1_opt = torch.mean(torch.cat(f1_25_list)).cpu().numpy(), torch.mean(torch.cat(f1_opt_list)).cpu().numpy()
    return mean_val_loss, mean_f1_25, mean_f1_opt

# Start training

In [ ]:
# print("schedulerCos:",schedulerCos.state_dict())
# print("schedulerMult:",schedulerMult.state_dict())
# for i in range(100):
#     print(i)
#     schedulerCos.step()
#     schedulerMult.step()
#     print("schedulerCos:",schedulerCos.state_dict())
#     print("schedulerMult:",schedulerMult.state_dict())

In [ ]:
model.eval()
with torch.no_grad():
    mean_val_loss, mean_f1_25, mean_f1_opt = validate()
print(f"Epoch {0}/{num_epochs}, train loss: NA; validation loss: {mean_val_loss:.6f}, F1-25: {mean_f1_25:.4f}, F1-opt: {mean_f1_opt:.4f}")

logdir = f"{datetime.now().strftime('%m%d_%H%M%S')}"
os.mkdir(logdir)
log_data = pd.DataFrame(index=np.arange(num_epochs), columns=["train_loss","val_loss","f1_25","f1_opt","filename"])
print(f"Training for {num_epochs} epochs started.")
for epoch in range(num_epochs):
    model.train()
    loss_array = np.zeros(len(train_loader))
    for batch_idx, (data_sentinel, data_landsat, data_cov, data_lonlat, targets, _) in tqdm(enumerate(train_loader), total=len(train_loader), desc="training", leave=False):
        data_sentinel = data_sentinel.to(device)[:,:,None,:,:]
        data_landsat, data_cov, data_lonlat, targets = data_landsat.to(device), data_cov.to(device), data_lonlat.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(data_sentinel, data_landsat, data_cov, data_lonlat)
        pos_weight = targets * positive_weigh_factor
        criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        loss = criterion(outputs, targets)
        loss_array[batch_idx] = loss.item()
        loss.backward()
        optimizer.step()
    mean_train_loss = np.mean(loss_array)
    
    model.eval()
    with torch.no_grad():
        mean_val_loss, mean_f1_25, mean_f1_opt = validate()

    cond1 = True if epoch < top_res_n else mean_val_loss < np.partition(log_data.val_loss[:epoch].values, top_res_n-1)[top_res_n-1]
    cond2 = True if epoch < top_res_n else -mean_f1_opt < np.partition(-log_data.f1_opt[:epoch].values, top_res_n-1)[top_res_n-1]
    print_suffix = "+++" if (epoch >= epoch_min) & (cond1 | cond2) else ""
    print(f"Epoch {epoch+1}/{num_epochs}, train loss: {mean_train_loss:.6f}; validation loss: {mean_val_loss:.6f}, F1-25: {mean_f1_25:.4f}, F1-opt: {mean_f1_opt:.4f}{print_suffix}")
    log_data.loc[epoch, ["train_loss","val_loss","f1_25","f1_opt"]] = [mean_train_loss, mean_val_loss, mean_f1_25, mean_f1_opt]
    if (epoch >= epoch_min) & (cond1 | cond2):
        res_filename = f"{datetime.now().strftime('%m%d_%H%M%S')}_e{epoch+1:03d}_vloss{mean_val_loss:.6f}_vf{mean_f1_opt:.4f}.csv"
        model.eval()
        with torch.no_grad():
            surveys, top_indices = [], [], []
            for data_sentinel, data_landsat, data_cov, data_lonlat, surveyId in tqdm(test_loader, total=len(test_loader),  desc="prediction", leave=False):
                data_sentinel = data_sentinel.to(device)[:,:,None,:,:]
                data_landsat, data_cov, data_lonlat = data_landsat.to(device), data_cov.to(device), data_lonlat.to(device)
                outputs = model(data_sentinel, data_landsat, data_cov, data_lonlat)
                top_batch_list = f1_score(outputs, None, device=device)
                top_batch_list = [np.sort(sp_categories[pred.cpu().numpy()]) for pred in top_batch_list]        
                surveyId = surveyId.numpy()
                top_indices += top_batch_list
                surveys.extend(surveyId)
        data_concatenated = [' '.join(map(str, row)) for row in top_indices]
        pd.DataFrame({'surveyId': surveys, 'predictions': data_concatenated,}).to_csv(os.path.join(logdir, res_filename), index=False)
        log_data.filename[epoch] = res_filename
    schedulerCos.step()
    # schedulerMult.step()
    # print("schedulerCos:",schedulerCos.state_dict())
    # print("schedulerMult:",schedulerMult.state_dict())
log_data.to_csv(os.path.join(logdir, "000_log.csv"))
epoch += 1

In [ ]:
log_data.to_csv(os.path.join(logdir, "000_log.csv"))
log_data.to_csv(os.path.join("logs", logdir+"_log.csv"))

In [ ]:
fig = plt.figure(figsize=[12,4])
ax = plt.subplot(1,2,1)
plt.plot(log_data.index, log_data.train_loss, label="train")
plt.plot(log_data.index, log_data.val_loss, label="val")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.legend()
ax = plt.subplot(1,2,2)
plt.plot(log_data.index, log_data.f1_opt, label="opt")
plt.plot(log_data.index, log_data.f1_25, label="top25")
ax.set_xlabel("epoch")
ax.set_ylabel("F1")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# Save the trained model
# model.eval()
# torch.save(model.state_dict(), logdir+f"_weights{num_epochs}.pth")

# Predict test data

In [ ]:
most_train_countries = (train_metadata.country.value_counts().cumsum() / test_metadata.shape[0]).index[:10].values
model.eval()
with torch.no_grad():
    surveys, top_indices = [], [], []
    for data_sentinel, data_landsat, data_cov, data_lonlat, surveyId in tqdm(test_loader, total=len(test_loader)):
        data_sentinel = data_sentinel.to(device)[:,:,None,:,:]
        data_landsat, data_cov, data_lonlat = data_landsat.to(device), data_cov.to(device), data_lonlat.to(device)
        outputs = model(data_sentinel, data_landsat, data_cov, data_lonlat)
        top_batch_list = f1_score(outputs, None, device=device)
        top_batch_list = [np.sort(sp_categorical.categories[pred.cpu().numpy()]) for pred in top_batch_list]        
        surveyId = surveyId.numpy()
        #tmp_metadata = test_metadata.loc[test_metadata.surveyId.isin(surveyId)]
        #tmp_metadata.set_index("surveyId", inplace=True)
        #tmp_metadata = tmp_metadata.loc[surveyId]
        #tmp_metadata.reset_index(drop=True, inplace=True)
        #tmp_metadata = tmp_metadata.loc[~tmp_metadata.country.isin(most_train_countries)]
        #for i in tmp_metadata.index:
        #    top_batch_list[i] = np.sort(train_metadata.value_counts("speciesId").index[:1000].values.astype(int))
        #    top_batch_list[i] = 11
        top_indices += top_batch_list
        surveys.extend(surveyId)

In [ ]:
data_concatenated = [' '.join(map(str, row)) for row in top_indices]
#timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
#pd.DataFrame({'surveyId': surveys, 'predictions': data_concatenated}).to_csv(f"{timestamp}_prithvi-frozen_bioresnet18_con5-elev-soil-drop05_regl2e4_e{epoch}_optlist.csv", index=False)